In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
print("\nTask 1: Reading the dataset...")
df = pd.read_csv('/root/.cache/kagglehub/datasets/mohammad2012191/q3-ka-ai-2026/versions/1/Q3_data.csv')
print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
# Task 2: Write your code here:
print("\n" + "="*80)
print("Task 2: First few rows of the dataset")
print("="*80)
print(df.head(10))

In [ ]:
# Task 3: Write your code here:
print("\n" + "="*80)
print("Task 3: Dataset Information")
print("="*80)
df.info()

In [ ]:
# Task 4: Write your code here:
print("\n" + "="*80)
print("Task 4: Statistical Description")
print("="*80)
print(df.describe())
# Additional exploration
print("\n" + "="*80)
print("ADDITIONAL DATA EXPLORATION")
print("="*80)

# Check target distribution
if 'target' in df.columns:
    target_col = 'target'
elif 'default' in df.columns:
    target_col = 'default'
else:
    # Find the target column (likely the last one or contains 'target'/'default')
    target_col = df.columns[-1]

print(f"\nTarget column identified: '{target_col}'")
print("\nTarget Distribution:")
print(df[target_col].value_counts())
print("\nTarget Proportions:")
print(df[target_col].value_counts(normalize=True))

# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
df[target_col].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_xlabel('Class', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Target Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xticklabels(['No Default (0)', 'Default (1)'], rotation=0)
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
df[target_col].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                    colors=['steelblue', 'coral'], startangle=90)
axes[1].set_ylabel('')
axes[1].set_title('Target Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your code here:
print("\nTask 1: Handling missing values...")
print("\nMissing values before handling:")
missing_before = df.isnull().sum()
print(missing_before[missing_before > 0])

if df.isnull().sum().sum() > 0:
    missing_pct = (df.isnull().sum() / len(df)) * 100
    missing_info = pd.DataFrame({
        'Missing_Count': df.isnull().sum(),
        'Percentage': missing_pct
    })
    print("\nMissing value details:")
    print(missing_info[missing_info['Missing_Count'] > 0].sort_values('Percentage', ascending=False))

    # Handle missing values
    for col in df.columns:
        if df[col].isnull().any() and col != target_col:
            if df[col].dtype in ['float64', 'int64']:
                # Fill numerical columns with median
                df[col].fillna(df[col].median(), inplace=True)
                print(f"Filled '{col}' with median")
            else:
                # Fill categorical columns with mode
                df[col].fillna(df[col].mode()[0], inplace=True)
                print(f"Filled '{col}' with mode")
else:
    print("No missing values found!")

print("\nMissing values after handling:")
print(df.isnull().sum().sum())

In [ ]:
# Task 2: Write your code here:
print("\n" + "="*80)
print("Task 2: Checking for duplicates...")
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Removed {duplicates} duplicate rows")
    print(f"Dataset shape after removing duplicates: {df.shape}")
else:
    print("No duplicates found!")

In [ ]:
# Task 3: Write your code here:
print("\n" + "="*80)
print("Task 3: Encoding categorical variables...")

# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if target_col in categorical_cols:
    categorical_cols.remove(target_col)

print(f"Categorical columns found: {categorical_cols}")

if categorical_cols:
    # Use Label Encoding for categorical columns
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
        print(f"Encoded '{col}' using LabelEncoder")
    print(f"Shape after encoding: {df.shape}")
else:
    print("No categorical columns to encode (all features are numerical - anonymized!)")


In [ ]:
# Task 4: Write your code here:
print("\n" + "="*80)
print("Task 4: Applying feature scaling...")

# Separate features and target
X = df.drop(target_col, axis=1)
y = df[target_col]

# Get feature names before scaling
feature_names = X.columns.tolist()

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_names)

print(f"Applied StandardScaler to {X_scaled.shape[1]} features")
print(f"Features shape: {X_scaled.shape}")
print(f"Target shape: {y.shape}")


In [ ]:
# Task 5: Write your code here:
print("\n" + "="*80)
print("Task 5: Checking for target imbalance...")

target_counts = y.value_counts()
target_props = y.value_counts(normalize=True)

print(f"\nTarget distribution:")
print(target_counts)
print(f"\nTarget proportions:")
print(target_props)

# Calculate imbalance ratio
minority_class = target_counts.min()
majority_class = target_counts.max()
imbalance_ratio = majority_class / minority_class

print(f"\nImbalance ratio: {imbalance_ratio:.2f}")

if imbalance_ratio > 1.5:
    print(f"  IMBALANCED DATASET DETECTED!")
    print(f"   Majority class: {majority_class} samples ({target_props.max()*100:.1f}%)")
    print(f"   Minority class: {minority_class} samples ({target_props.min()*100:.1f}%)")
    print(f"   This is a classification problem with class imbalance.")
    print(f"   We should use F1-Score instead of Accuracy for evaluation!")
else:
    print(f"  Balanced dataset - both classes are well represented")

In [ ]:
# Task 1: Write your code here:
print("\nTask 1: Features (X) and Target (y) split completed")
print(f"X shape: {X_scaled.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
print("\n" + "="*80)
print("Tasks 2-5: StratifiedKFold Cross-Validation with CatBoostClassifier")
print("="*80)
# Use StratifiedKFold for imbalanced classification
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracy_scores = []
f1_scores = []
fold_num = 1

# Store feature importance from each fold
feature_importance_list = []

print("\nTraining CatBoostClassifier with 5-fold cross-validation...\n")

for train_idx, val_idx in skfold.split(X_scaled, y):
    # Split data
    X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train CatBoost model
    catboost_model = CatBoostClassifier(
        iterations=100,
        learning_rate=0.1,
        depth=6,
        random_state=42,
        verbose=0
    )
    catboost_model.fit(X_train, y_train)

    # Predict
    y_pred = catboost_model.predict(X_val)

    # Evaluate using both Accuracy and F1 Score
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    accuracy_scores.append(acc)
    f1_scores.append(f1)

    # Store feature importance
    feature_importance_list.append(catboost_model.feature_importances_)

    print(f"Fold {fold_num}: Accuracy = {acc:.4f} | F1-Score = {f1:.4f}")
    fold_num += 1

print("-"*80)

# Determine which metric to use based on imbalance
if imbalance_ratio > 1.5:
    primary_metric = "F1-Score"
    primary_scores = f1_scores
    print(f"\n PRIMARY METRIC: F1-Score (dataset is imbalanced)")
else:
    primary_metric = "Accuracy"
    primary_scores = accuracy_scores
    print(f"\n PRIMARY METRIC: Accuracy (dataset is balanced)")

print(f"\nAveraged {primary_metric} across all folds: {np.mean(primary_scores):.4f} ± {np.std(primary_scores):.4f}")
print(f"Averaged Accuracy across all folds: {np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}")
print(f"Averaged F1-Score across all folds: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")

# Train final model on full data for feature importance
print("\nTraining final model on full dataset for feature analysis...")
final_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    random_state=42,
    verbose=0
)
final_model.fit(X_scaled, y)


In [ ]:
# Task 1: Write your code here:
print("\nTask 1: Plotting feature importance...")

# Get average feature importance across all folds

avg_feature_importance = np.mean(feature_importance_list, axis=0)

feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': avg_feature_importance
}).sort_values('importance', ascending=False)

# Plot feature importance
plt.figure(figsize=(14, max(8, len(feature_names) * 0.3)))
colors = plt.cm.viridis(np.linspace(0, 1, len(feature_importance_df)))
colors[0] = [1, 0.84, 0, 1]  # Gold color for the top feature

bars = plt.barh(range(len(feature_importance_df)),
                feature_importance_df['importance'],
                color=colors)
plt.yticks(range(len(feature_importance_df)), feature_importance_df['feature'])
plt.xlabel('Importance Score', fontsize=12, fontweight='bold')
plt.ylabel('Features', fontsize=12, fontweight='bold')
plt.title('Feature Importance from CatBoost Model\n(Golden Feature Highlighted in Gold)',
          fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
print("\n" + "="*80)
print("Task 2: Identifying the Golden Feature...")
print("="*80)

golden_feature = feature_importance_df.iloc[0]['feature']
golden_importance = feature_importance_df.iloc[0]['importance']

print(f"\n THE GOLDEN FEATURE IS: '{golden_feature}'")
print(f"   Importance Score: {golden_importance:.4f}")
print(f"\n This feature is the most powerful predictor of credit default!")

print("\nTop 10 Most Important Features:")
print(feature_importance_df.head(10).to_string(index=False))

In [ ]:
# Task Bonus: Write your code here:
X_golden = X_scaled[[golden_feature]]
print(f"Golden feature shape: {X_golden.shape}")

# Task 2 & 3: Run KFold with single feature
accuracy_scores_golden = []
f1_scores_golden = []
fold_num = 1

print("\nTraining with golden feature only...\n")

for train_idx, val_idx in skfold.split(X_golden, y):
    # Split data
    X_train_golden, X_val_golden = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train CatBoost model with single feature
    catboost_golden = CatBoostClassifier(
        iterations=100,
        learning_rate=0.1,
        depth=6,
        random_state=42,
        verbose=0
    )
    catboost_golden.fit(X_train_golden, y_train)

    # Predict
    y_pred_golden = catboost_golden.predict(X_val_golden)

    # Evaluate
    acc_golden = accuracy_score(y_val, y_pred_golden)
    f1_golden = f1_score(y_val, y_pred_golden)

    accuracy_scores_golden.append(acc_golden)
    f1_scores_golden.append(f1_golden)

    print(f"Fold {fold_num}: Accuracy = {acc_golden:.4f} | F1-Score = {f1_golden:.4f}")
    fold_num += 1

print("-"*80)

# Comparison
print("\n" + "="*80)
print("PERFORMANCE COMPARISON: FULL MODEL VS GOLDEN FEATURE ONLY")
print("="*80)

avg_acc_full = np.mean(accuracy_scores)
avg_acc_golden = np.mean(accuracy_scores_golden)
avg_f1_full = np.mean(f1_scores)
avg_f1_golden = np.mean(f1_scores_golden)

print(f"\n FULL MODEL (All {len(feature_names)} features):")
print(f"   Accuracy:  {avg_acc_full:.4f}")
print(f"   F1-Score:  {avg_f1_full:.4f}")

print(f"\n🏆 GOLDEN FEATURE ONLY ('{golden_feature}'):")
print(f"   Accuracy:  {avg_acc_golden:.4f}")
print(f"   F1-Score:  {avg_f1_golden:.4f}")

print(f"\n PERFORMANCE DIFFERENCE:")
print(f"   Accuracy Drop:  {(avg_acc_full - avg_acc_golden):.4f} ({((avg_acc_full - avg_acc_golden)/avg_acc_full)*100:.2f}%)")
print(f"   F1-Score Drop:  {(avg_f1_full - avg_f1_golden):.4f} ({((avg_f1_full - avg_f1_golden)/avg_f1_full)*100:.2f}%)")

# Performance retention
acc_retention = (avg_acc_golden / avg_acc_full) * 100
f1_retention = (avg_f1_golden / avg_f1_full) * 100

print(f"\n INSIGHT:")
print(f"   The golden feature alone retains {acc_retention:.1f}% of the full model's accuracy")
print(f"   The golden feature alone retains {f1_retention:.1f}% of the full model's F1-score")

if acc_retention > 85:
    print(f"\n    Amazing! The golden feature '{golden_feature}' is incredibly powerful!")
    print(f"      Using just this one feature achieves {acc_retention:.1f}% of full model performance!")
elif acc_retention > 70:
    print(f"\n   ✓ The golden feature '{golden_feature}' is quite strong!")
    print(f"     It captures most of the predictive power on its own.")
else:
    print(f"\n   The golden feature '{golden_feature}' is important but works best with other features.")

# Visualization of comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart comparison
metrics = ['Accuracy', 'F1-Score']
full_model_scores = [avg_acc_full, avg_f1_full]
golden_scores = [avg_acc_golden, avg_f1_golden]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, full_model_scores, width, label='Full Model', color='steelblue')
axes[0].bar(x + width/2, golden_scores, width, label='Golden Feature Only', color='gold')
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, v in enumerate(full_model_scores):
    axes[0].text(i - width/2, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')
for i, v in enumerate(golden_scores):
    axes[0].text(i + width/2, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

# Feature count vs performance
axes[1].plot([len(feature_names), 1], [avg_acc_full, avg_acc_golden],
             'o-', linewidth=2, markersize=10, label='Accuracy', color='steelblue')
axes[1].plot([len(feature_names), 1], [avg_f1_full, avg_f1_golden],
             's-', linewidth=2, markersize=10, label='F1-Score', color='coral')
axes[1].set_xlabel('Number of Features', fontsize=12)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Performance vs Feature Count', fontsize=14, fontweight='bold')
axes[1].set_xticks([1, len(feature_names)])
axes[1].set_xticklabels(['1\n(Golden Only)', f'{len(feature_names)}\n(All Features)'])
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("MISSION COMPLETE! ")
print("="*80)
print(f"The golden feature '{golden_feature}' has been successfully identified!")
print("="*80)